In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Installation
# ══════════════════════════════════════════════════════════════════════════════
import subprocess

def _run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {label or cmd[:60]}")
    if r.returncode != 0:
        print(r.stderr[-300:])
    return r.returncode == 0

_run("apt-get install -y -q tesseract-ocr tesseract-ocr-ara poppler-utils",
     "Tesseract OCR arabe + Poppler")
_run("pip install -q pytesseract Pillow pymupdf",           "pytesseract + Pillow + PyMuPDF")
_run("pip install -q transformers sentence-transformers",   "Sentence-Transformers")
_run("pip install -q torch --index-url https://download.pytorch.org/whl/cu118", "PyTorch CUDA")
_run("pip install -q rank_bm25",                            "BM25")

if not _run("pip install -q faiss-gpu-cu11", "FAISS GPU"):
    _run("pip install -q faiss-cpu", "FAISS CPU (fallback)")

_run("pip install -q llama-cpp-python "
     "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122",
     "llama-cpp CUDA")
_run("pip install -q huggingface_hub tqdm ipywidgets", "HuggingFace Hub + widgets")
print("\n Installation complète.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Extraction OCR (inchangée)
# ══════════════════════════════════════════════════════════════════════════════
import os, json, glob, re, time, subprocess, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

OCR_CACHE      = "/content/loc_ocr_pages.json"
ARTICLES_CACHE = "/content/loc_articles.json"
EMBED_CACHE    = "/content/loc_embeddings.npy"
IMG_DIR        = "/content/loc_pages"
os.makedirs(IMG_DIR, exist_ok=True)

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    PDF_PATH = "/content/drive/MyDrive/3D_SMART/lois/les_lois_des_obligations_et_contrats.pdf"
    if not os.path.exists(PDF_PATH):
        print("PDF non trouvé sur Drive — upload manuel :")
        uploaded = files.upload()
        PDF_PATH = f"/content/{list(uploaded.keys())[0]}"
except Exception:
    PDF_PATH = "/content/loc.pdf"

print(f"PDF : {PDF_PATH}")
assert os.path.exists(PDF_PATH), f"Fichier introuvable : {PDF_PATH}"


def rasterize_pdf(pdf_path, out_dir, dpi=300):
    existing = sorted(glob.glob(f"{out_dir}/page-*.jpg"))
    if existing:
        print(f"Images déjà présentes : {len(existing)} pages")
        return existing
    print(f"Rasterisation à {dpi} DPI…")
    subprocess.run(["pdftoppm", "-jpeg", "-r", str(dpi), pdf_path, f"{out_dir}/page"], check=True)
    pages = sorted(glob.glob(f"{out_dir}/page-*.jpg"))
    print(f"{len(pages)} pages rasterisées")
    return pages


def _ocr_worker(args):
    from PIL import Image
    import pytesseract
    img_path, page_num = args
    try:
        text = pytesseract.image_to_string(
            Image.open(img_path), lang="ara",
            config="--psm 6 --oem 3 -c preserve_interword_spaces=1",
        )
        return page_num, text.strip()
    except Exception as exc:
        return page_num, f"[OCR_ERROR page {page_num}: {exc}]"


def run_ocr(pdf_path, img_dir, cache_path):
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        pages = {int(k): v for k, v in cached.items()}
        avg = sum(len(v) for v in pages.values()) / max(len(pages), 1)
        if len(pages) >= 10 and avg >= 200:
            print(f"Cache OCR valide : {len(pages)} pages (moy. {avg:.0f} chars/page)")
            return pages
        os.remove(cache_path)

    page_files = rasterize_pdf(pdf_path, img_dir)
    n_workers = min(multiprocessing.cpu_count(), 8)
    args = [(f, i + 1) for i, f in enumerate(page_files)]
    results = {}

    try:
        from tqdm.notebook import tqdm
    except ImportError:
        from tqdm import tqdm

    t0 = time.time()
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(_ocr_worker, a): a[1] for a in args}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="OCR Tesseract"):
            pn, txt = fut.result()
            results[pn] = txt

    print(f"OCR terminé en {(time.time()-t0)/60:.1f} min")
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results


ocr_pages = run_ocr(PDF_PATH, IMG_DIR, OCR_CACHE)
print(f"Aperçu page 5 :\n{ocr_pages.get(5,'')[:400]}")


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Nettoyage OCR (CORRIGÉ — bug item "3 -" résolu)
# ══════════════════════════════════════════════════════════════════════════════

_HEADER_RE = re.compile(
    r"(?:(?:المملكة|المغرب)[^\n]*(?:وزارة|مديرية|مكيرية)[^\n]*"
    r"|(?:وزارة|مديرية)[^\n]*(?:التشريع)[^\n]*)",
    re.UNICODE,
)
_PAGE_NUM_RE = re.compile(r"^[\-\–\s]*\d{1,3}[\-\–\s]*$")

# ── CORRECTION : verbes de référence légale EXPLICITES uniquement ────────────
# On n'active le mode "note de bas de page" QUE si ces verbes sont présents.
# Cela préserve les items de liste "3 - موضوع الالتزام" dans les articles.
_NOTE_VERBS_RE = re.compile(
    r"""(?:
        ورد\s+في\s+النص       |  # "ورد في النص" = renvoi textuel
        وردت\s+في\s+النص      |
        قارن\s+مع             |  # "قارن مع X"
        انظر\s+(?:الفصل|المادة|البند)  |  # "انظر الفصل N"
        تمم\s+(?:بمقتضى|الفصل)        |  # "تمم بمقتضى"
        تممت\s+(?:بمقتضى|الفصل)       |
        أضاف\s+(?:بمقتضى|الفصل)       |
        أضيف\s+(?:بمقتضى|الفصل)       |
        أُلغي\s+(?:بمقتضى|الفصل)      |
        سقط\s+(?:بمقتضى|الفصل)        |
        استُبدل\s+(?:بمقتضى|الفصل)    |
        راجع\s+(?:الفصل|المادة)
    )""",
    re.VERBOSE | re.UNICODE,
)

# Citation officielle isolée (Journal officiel, Dahir numéroté)
_OFFICIAL_CITE_RE = re.compile(
    r"""(?:
        الجريدة\s+الرسمية\s+(?:عدد|رقم)   |
        ظهير\s+شريف\s+رقم\s+\d            |
        بتنفيذه\s+ظهير\s+شريف              |
        صادر\s+في\s+\d+\s+(?:من|يناير|فبراير|مارس|أبريل|ماي|يونيو|يوليوز|غشت|شتنبر|أكتوبر|نونبر|دجنبر) |
        عدد\s+\d{4,5}\s+(?:بتاريخ|مكرر)   |
        ص\.\s*\d{3,}                        |
        القانون\s+رقم\s+\d+\.\d+\s+الصادر  |
        رقم\s+\d+\.\d+\.\d+\s+الصادر
    )""",
    re.VERBOSE | re.UNICODE,
)

_ARTICLE_OR_TITLE_RE = re.compile(
    r"^(?:الفصل\s+\d|الكتاب\s+|القسم\s+|الباب\s+|الفرع\s+)",
    re.UNICODE,
)

# Item de liste INTERNE à un article : "N - texte_arabe_court"
# Ces lignes doivent TOUJOURS être conservées
_INTERNAL_LIST_RE = re.compile(
    r"^\d{1,2}\s*[-–]\s*[\u0600-\u06FF]",
    re.UNICODE,
)


def clean_page(raw_text):
    """
    Nettoie une page OCR.

    RÈGLE FONDAMENTALE (CORRIGÉE) :
    Une ligne "N - ..." est supprimée comme note de bas de page SEULEMENT si :
      a) Elle contient un verbe de référence légal explicite (_NOTE_VERBS_RE)
      b) OU elle contient une citation officielle ET n'est pas un item de liste court

    Sinon, elle est conservée (items de liste légitimes dans les articles).
    """
    lines = raw_text.split("\n")
    cleaned = []
    in_footnote = False

    for line in lines:
        s = line.strip()

        if not s:
            if not in_footnote:
                cleaned.append("")
            continue

        if _PAGE_NUM_RE.match(s):
            continue

        if _HEADER_RE.search(s):
            continue

        # Début d'article ou titre structurel → quitter le mode note
        if _ARTICLE_OR_TITLE_RE.match(s):
            in_footnote = False

        # ── Détection de note (LOGIQUE CORRIGÉE) ────────────────────────────
        if not in_footnote:
            starts_num_dash = re.match(r"^\d{1,3}\s*[-–]", s)

            if starts_num_dash:
                has_ref_verb = bool(_NOTE_VERBS_RE.search(s))
                has_official = bool(_OFFICIAL_CITE_RE.search(s))
                # Item de liste court sans verbe de référence → CONSERVER
                is_list_item = bool(_INTERNAL_LIST_RE.match(s)) and len(s) < 80

                if has_ref_verb or (has_official and not is_list_item):
                    in_footnote = True
                # Sinon : item de liste légitime → conserver

            elif _OFFICIAL_CITE_RE.search(s) and len(s) < 120:
                # Citation officielle seule sur une ligne → note
                in_footnote = True

        if in_footnote:
            continue

        cleaned.append(s)

    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# Vérification que les 4 items du الفصل 2 sont présents après nettoyage
print("--- Vérification page 5 (الفصل 2 doit avoir items 1,2,3,4) ---")
p5_clean = clean_page(ocr_pages.get(5, ""))
print(p5_clean[:600])



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Parsing des articles
# ══════════════════════════════════════════════════════════════════════════════
from dataclasses import dataclass, asdict

LOC_MAX_NUM = 1260


@dataclass
class LegalArticle:
    article_num:     str
    article_num_int: int
    text:            str
    book:            str = ""
    section:         str = ""
    subsection:      str = ""
    char_count:      int = 0

    def to_display(self):
        lines = [f"{'='*60}", f"الفصل {self.article_num}"]
        if self.book:       lines.append(f"[{self.book}]")
        if self.section:    lines.append(f"[{self.section}]")
        if self.subsection: lines.append(f"[{self.subsection}]")
        lines += ["", self.text, ""]
        return "\n".join(lines)

    def to_chunk_text(self):
        hdr = f"[الفصل {self.article_num}]"
        if self.book:       hdr += f" [{self.book}]"
        if self.section:    hdr += f" [{self.section}]"
        if self.subsection: hdr += f" [{self.subsection}]"
        return f"{hdr}\n{self.text}"


def _num_to_int(s):
    s = s.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    m = re.search(r"\d+", s)
    return int(m.group()) if m else 0


def _clean_article_num(raw):
    cleaned = re.sub(r"[^\d.\-]", "", raw)
    first = _num_to_int(cleaned)
    if first == 0 or first > LOC_MAX_NUM:
        digits = re.match(r"(\d+)", cleaned)
        if digits:
            n = digits.group(1)
            for l in range(len(n), 0, -1):
                if 0 < int(n[:l]) <= LOC_MAX_NUM:
                    rest = cleaned[len(n):]
                    return n[:l] + rest if rest else n[:l]
    return cleaned


class LOCParser:
    # الفصل suivi d'UN ESPACE puis d'un numéro (évite de matcher "2 - الأهلية")
    ART_SPLIT = re.compile(
        r"(?:^|\n)(الفصل\s+\d+(?:[.\-]\d+)?)\s*\n",
        re.MULTILINE | re.UNICODE,
    )
    ART_NUM = re.compile(r"الفصل\s+(\d+(?:[.\-]\d+)?)", re.UNICODE)

    BOOK_RE = re.compile(r"(الكتاب\s+\S+(?:\s+(?:من|في|ل)\s+\S+)?)", re.UNICODE)
    SECT_RE = re.compile(r"((?:القسم|الباب)\s+\S+(?:\s+\S+)?)", re.UNICODE)
    SUBS_RE = re.compile(r"(الفرع\s+\S+(?:\s+\S+)?)", re.UNICODE)

    # Superscript : chiffre COLLÉ à la fin d'un mot arabe (pas un item de liste)
    SUPERSCRIPT_RE = re.compile(
        r"([\u0600-\u06FF])\d{1,3}(?=[\s،؛\.\n»\u00BB]|$)",
        re.UNICODE,
    )
    PAGE_SEP_RE = re.compile(r"---\s*PAGE\s+\d+\s*---\n?")

    def _clean_body(self, raw_body):
        text = self.PAGE_SEP_RE.sub("", raw_body)
        text = self.SUPERSCRIPT_RE.sub(r"\1", text)
        text = re.sub(r'[""\u201c\u201d]', "", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def _update_context(self, text, book, sect, subs):
        bm = self.BOOK_RE.search(text)
        sm = self.SECT_RE.search(text)
        sb = self.SUBS_RE.search(text)
        return (
            bm.group(1).strip() if bm else book,
            sm.group(1).strip() if sm else sect,
            sb.group(1).strip() if sb else subs,
        )

    def parse(self, ocr_pages):
        parts_text = []
        for pg in sorted(ocr_pages.keys(), key=int):
            cleaned = clean_page(ocr_pages[pg])
            parts_text.append(f"--- PAGE {pg} ---\n{cleaned}")
        full_text = "\n".join(parts_text)

        print(f"Texte total : {len(full_text):,} chars | {full_text.count(chr(10)):,} lignes")
        n_matches = len(self.ART_SPLIT.findall(full_text))
        print(f"Marqueurs 'الفصل N' détectés : {n_matches}")

        if n_matches == 0:
            raise ValueError("Aucun article trouvé — vérifiez l'OCR.")

        parts = self.ART_SPLIT.split(full_text)
        print(f"Split : {len(parts)} parties → ~{(len(parts)-1)//2} articles bruts")

        articles = []
        cur_book = cur_sect = cur_subs = ""
        n_skipped = 0

        if parts:
            cur_book, cur_sect, cur_subs = self._update_context(parts[0], cur_book, cur_sect, cur_subs)

        i = 1
        while i < len(parts):
            header = parts[i].strip()
            body   = parts[i + 1].strip() if (i + 1) < len(parts) else ""
            i += 2

            cur_book, cur_sect, cur_subs = self._update_context(body, cur_book, cur_sect, cur_subs)

            nm = self.ART_NUM.search(header)
            if not nm:
                n_skipped += 1
                continue

            num_str = _clean_article_num(nm.group(1))
            num_int = _num_to_int(num_str)

            if num_int == 0 or num_int > LOC_MAX_NUM:
                n_skipped += 1
                continue

            clean_body = self._clean_body(body)
            if len(clean_body) < 25:
                n_skipped += 1
                continue

            articles.append(LegalArticle(
                article_num=num_str, article_num_int=num_int,
                text=clean_body, book=cur_book, section=cur_sect,
                subsection=cur_subs, char_count=len(clean_body),
            ))

        print(f"Articles extraits (avant dédup) : {len(articles)}")
        print(f"Partitions ignorées             : {n_skipped}")

        seen = {}
        for art in articles:
            if art.article_num not in seen or art.char_count > seen[art.article_num].char_count:
                seen[art.article_num] = art

        result = sorted(seen.values(), key=lambda a: (a.article_num_int, a.article_num))
        print(f"Articles uniques après dédup    : {len(result)}")
        return result

    def save(self, articles, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump([asdict(a) for a in articles], f, ensure_ascii=False, indent=2)
        print(f"{len(articles)} articles sauvegardés -> {path}")

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            return [LegalArticle(**d) for d in json.load(f)]


parser = LOCParser()
articles = []

if os.path.exists(ARTICLES_CACHE):
    with open(ARTICLES_CACHE, "r", encoding="utf-8") as _f:
        _data = json.load(_f)
    if _data:
        articles = parser.load(ARTICLES_CACHE)
        print(f"Cache articles : {len(articles)} articles")
    else:
        os.remove(ARTICLES_CACHE)

if not articles:
    articles = parser.parse(ocr_pages)
    if articles:
        parser.save(articles, ARTICLES_CACHE)
    else:
        raise RuntimeError("Aucun article extrait.")

chars = [a.char_count for a in articles]
print(f"\nTotal articles : {len(articles)}")
print(f"Plage : الفصل {articles[0].article_num} -> الفصل {articles[-1].article_num}")
print(f"Chars/article : min={min(chars)} | moy={sum(chars)//len(chars)} | max={max(chars)}")

# ── VÉRIFICATION الفصل 2 ─────────────────────────────────────────────────────
art2 = next((a for a in articles if a.article_num == "2"), None)
if art2:
    print(f"\nVÉRIFICATION الفصل 2 :\n{art2.text[:400]}")
    for item in ["1 -", "2 -", "3 -", "4 -"]:
        present = item in art2.text
        print(f"  Item {item} : {'PRESENT' if present else 'MANQUANT !!!'}")
else:
    print("ATTENTION : الفصل 2 introuvable dans la base !")


In [ ]:


# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Embeddings + Index FAISS (inchangée)
# ══════════════════════════════════════════════════════════════════════════════
import torch, numpy as np, faiss
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

EMBED_MODELS = [
    "CAMeL-Lab/camel-bert-base-sts",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

embed_model = None
for mid in EMBED_MODELS:
    try:
        print(f"Chargement : {mid}…")
        embed_model = SentenceTransformer(mid, device=DEVICE)
        print(f"OK : {mid}")
        break
    except Exception as e:
        print(f"Echec : {str(e)[:80]}")

if embed_model is None:
    raise RuntimeError("Aucun modèle d'embedding disponible.")

chunk_texts = [a.to_chunk_text() for a in articles]

embeddings = None
if os.path.exists(EMBED_CACHE):
    embeddings = np.load(EMBED_CACHE)
    if embeddings.shape[0] != len(articles):
        os.remove(EMBED_CACHE)
        embeddings = None

if embeddings is None:
    print(f"Encodage de {len(chunk_texts)} articles…")
    embeddings = embed_model.encode(
        chunk_texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    np.save(EMBED_CACHE, embeddings)

DIM = embeddings.shape[1]
cpu_index = faiss.IndexFlatIP(DIM)
cpu_index.add(embeddings.astype(np.float32))

if DEVICE == "cuda":
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    print(f"FAISS GPU : {index.ntotal} vecteurs x {DIM} dims")
else:
    index = cpu_index
    print(f"FAISS CPU : {index.ntotal} vecteurs x {DIM} dims")


In [ ]:


# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — BM25 + HybridRetriever AVEC SEUIL RRF (CORRIGÉ)
#
# NOUVEAUTÉ : retrieve_filtered() élimine les articles hors-sujet via un
# seuil RRF minimum. Plus de الفصل 958 (gestion d'affaires), الفصل 600
# (vente Thunya), ou الفصل 37 (aveu) injectés pour une question sur la
# durée du contrat de travail.
# ══════════════════════════════════════════════════════════════════════════════
from rank_bm25 import BM25Okapi
from dataclasses import dataclass as _dc

STOP_WORDS = {
    "من", "في", "على", "عن", "إلى", "هذا", "هذه", "التي", "الذي", "وأن",
    "أن", "لا", "ما", "مع", "أو", "ولا", "لم", "كان", "يكون", "ذلك",
    "هو", "هي", "إذا", "إذ", "عند", "بعد", "قبل", "حتى", "إن", "كل",
    "غير", "بين", "كما", "حين", "منذ", "لكن", "ثم", "لدى",
}

# ── Seuil RRF : explication ──────────────────────────────────────────────────
# Score RRF maximum théorique (rang 1 dans dense ET sparse) = 1/(60+1)*2 ≈ 0.0328
# Un article pertinent devrait avoir RRF ≥ 0.012 (≈ 37% du max).
# En dessous, l'article est probablement un faux positif.
RRF_MIN_THRESHOLD = 0.012


def tokenize_ar(text):
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s\d]", " ", text)
    return [t for t in text.split() if len(t) > 2 and t not in STOP_WORDS]


tokenized = [tokenize_ar(a.to_chunk_text()) for a in articles]
bm25 = BM25Okapi(tokenized)

article_index = {a.article_num: i for i, a in enumerate(articles)}
for i, a in enumerate(articles):
    article_index.setdefault(str(a.article_num_int), i)

print(f"BM25 : {len(tokenized)} documents indexés")
print(f"Index direct : {len(article_index)} entrées")


@_dc
class RetrievedArticle:
    article:     LegalArticle
    dense_score: float
    bm25_score:  float
    rrf_score:   float
    dense_rank:  int
    bm25_rank:   int


class HybridRetriever:
    """
    Retrieval hybride Dense+BM25+RRF avec filtrage de pertinence.

    Méthodes :
      retrieve(query, top_k)         : top-K sans seuil (debug uniquement)
      retrieve_filtered(query, top_k): top-K avec seuil RRF (production)
    """

    def __init__(self, arts, faiss_idx, emb_model, bm25_model, k=60):
        self.arts  = arts
        self.index = faiss_idx
        self.embed = emb_model
        self.bm25  = bm25_model
        self.k     = k

    def _dense(self, query, n=20):
        vec = self.embed.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        scores, ids = self.index.search(vec, n)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def _sparse(self, query, n=20):
        tokens = tokenize_ar(query)
        if not tokens:
            return []
        sc = self.bm25.get_scores(tokens)
        top = np.argsort(sc)[::-1][:n]
        return [(int(i), float(sc[i])) for i in top]

    def _rrf(self, dense_res, sparse_res, n=10):
        rrf_sc = {}
        d_rank, b_rank, d_sc, b_sc = {}, {}, {}, {}
        for rk, (i, s) in enumerate(dense_res):
            rrf_sc[i] = rrf_sc.get(i, 0) + 1.0 / (self.k + rk + 1)
            d_rank[i] = rk + 1; d_sc[i] = s
        for rk, (i, s) in enumerate(sparse_res):
            rrf_sc[i] = rrf_sc.get(i, 0) + 1.0 / (self.k + rk + 1)
            b_rank[i] = rk + 1; b_sc[i] = s
        top = sorted(rrf_sc, key=rrf_sc.get, reverse=True)[:n]
        return [
            RetrievedArticle(
                article=self.arts[i],
                dense_score=d_sc.get(i, 0.0),
                bm25_score=b_sc.get(i, 0.0),
                rrf_score=rrf_sc[i],
                dense_rank=d_rank.get(i, 999),
                bm25_rank=b_rank.get(i, 999),
            )
            for i in top if 0 <= i < len(self.arts)
        ]

    def retrieve(self, query, top_k=5):
        """Sans seuil (usage debug)."""
        return self._rrf(self._dense(query, 20), self._sparse(query, 20), top_k)

    def retrieve_filtered(self, query, top_k=5, min_threshold=RRF_MIN_THRESHOLD):
        """
        CORRIGÉ : retourne uniquement les articles avec RRF >= seuil.

        Algorithme :
          1. Récupérer top_k*2 candidats
          2. Filtrer ceux avec RRF < min_threshold
          3. Si aucun passe le seuil → fallback top-3 (questions très spécifiques)
          4. Retourner au maximum top_k articles pertinents
        """
        candidates = self._rrf(
            self._dense(query, 30),
            self._sparse(query, 30),
            top_k * 2
        )
        filtered = [r for r in candidates if r.rrf_score >= min_threshold]

        if not filtered:
            print(f"   Aucun article au-dessus du seuil {min_threshold:.4f} — fallback top-3")
            return candidates[:3]

        return filtered[:top_k]


retriever = HybridRetriever(articles, index, embed_model, bm25)

# Test avec la question de l'exemple
print("\nTest retrieval filtre : 'مدة العقد غير محددة عقد العمل'")
for r in retriever.retrieve_filtered("مدة العقد غير محددة عقد العمل", 6):
    print(f"  الفصل {r.article.article_num:>6} | RRF={r.rrf_score:.4f} | {r.article.text[:55]}…")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — LLM Qwen2.5-7B-Instruct (inchangée)
# ══════════════════════════════════════════════════════════════════════════════
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODEL_REPO = "bartowski/Qwen2.5-7B-Instruct-GGUF"
MODEL_FILE = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
MODEL_PATH = f"/content/{MODEL_FILE}"

if not os.path.exists(MODEL_PATH):
    print(f"Téléchargement {MODEL_FILE} (~4.4 GB)…")
    hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/content")
else:
    print(f"Modèle déjà présent : {MODEL_PATH}")

llm = Llama(
    model_path=MODEL_PATH, n_gpu_layers=-1, n_ctx=4096,
    n_batch=512, verbose=False, seed=42, chat_format="chatml",
)
print("Qwen2.5-7B-Instruct chargé")

_t = llm.create_chat_completion(
    messages=[{"role": "user", "content": "قل: جاهز"}], max_tokens=10
)
print(f"Test LLM : {_t['choices'][0]['message']['content']}")


In [ ]:


# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Agent juridique (CORRIGÉ)
#
# CORRECTIONS :
#   1. retrieve_filtered() au lieu de retrieve() → articles hors-sujet éliminés
#   2. Prompt système renforcé → instruction d'ignorer les articles non liés
#   3. Extraction des فصول cités par le LLM → affichage ciblé des textes officiels
# ══════════════════════════════════════════════════════════════════════════════

class LawConsultantAgent:
    """
    Agent RAG pour le LOC marocain — version corrigée.

    Flux de traitement :
      1. Lookup direct si la question cite "الفصل N"
      2. retrieve_filtered() : retrieval sémantique AVEC seuil RRF
      3. Fusion sans doublons (directs prioritaires)
      4. Génération LLM avec prompt renforcé
      5. Extraction des فصول réellement cités par le LLM
      6. Affichage ciblé des textes officiels
    """

    # ── Prompt renforcé ───────────────────────────────────────────────────────
    SYSTEM_PROMPT = (
    "أنت مستشار قانوني مغربي خبير في قانون الالتزامات والعقود (LOC)، "
    "وتعتمد على تفسير قانوني دقيق للنصوص وليس مجرد نقلها حرفيا.\n\n"

    "🎯 المهمة:\n"
    "الإجابة على الأسئلة القانونية بالاعتماد فقط على الفصول المقدمة، "
    "مع تفسير قانوني صحيح يعكس المعنى الحقيقي للنص.\n\n"

    "⚖️ قواعد صارمة:\n"
    "1. استند فقط على الفصول المقدمة المرتبطة مباشرة بالسؤال.\n"
    "2. تجاهل أي فصل غير مرتبط مباشرة.\n"
    "3. ابدأ بذكر الفصول المستعملة (مثال: وفق الفصل 754...).\n"
    "4. اقتبس الجزء المهم فقط من النص (ليس النص الكامل).\n"
    "5. قدم تفسيرا قانونيا دقيقا وليس مجرد إعادة صياغة.\n"
    "6. لا تضف أي معلومات غير موجودة في النص.\n"
    "7. إذا لم توجد إجابة في الفصول، صرح بذلك بوضوح.\n\n"

    "🧠 قواعد التفسير القانوني (مهمة جدا):\n"
    "- لا تفسر المصطلحات القانونية حرفيا إذا كان لها معنى خاص في القانون.\n"
    "- يجب تفسير النص وفق سياقه القانوني.\n"
    "- مثال مهم:\n"
    "  • 'قابل للإبطال' لا تعني دائما أن العقد غير صحيح.\n"
    "  • في عقود العمل: قد تعني أن العقد صحيح لكنه قابل للإنهاء بإرادة أحد الطرفين.\n"
    "- إذا كان النص يتحدث عن إمكانية إنهاء العقد، ففسر ذلك كحق في الفسخ وليس بطلان العقد.\n\n"

    "⚠️ تجنب الأخطاء التالية:\n"
    "- لا تعتبر العقد باطلا إلا إذا نص القانون صراحة على ذلك.\n"
    "- لا تخلط بين: البطلان / الإبطال / الفسخ / الانتهاء.\n"
    "- لا تقدم تفسيرا يخالف منطوق النص.\n\n"

    "📌 شكل الجواب:\n"
    "1. الفصول القانونية المستعملة\n"
    "2. مقتطف من النص\n"
    "3. التفسير القانوني الدقيق\n"
)

    _ART_REF_RE = re.compile(r"الفصل\s*(\d+(?:[.\-]\d+)?)", re.UNICODE)

    # Extraire les numéros de فصول cités dans la réponse du LLM
    _CITED_ART_RE = re.compile(
        r"(?:الفصل|وفق\s+الفصل|بمقتضى\s+الفصل|نص\s+الفصل|طبقاً\s+للفصل)\s+(\d+(?:[.\-]\d+)?)",
        re.UNICODE,
    )

    def __init__(self, retriever, llm, articles, article_index):
        self.retriever     = retriever
        self.llm           = llm
        self.articles      = articles
        self.article_index = article_index

    def _lookup_direct(self, question):
        found = []
        for raw in self._ART_REF_RE.findall(question):
            num = _clean_article_num(raw)
            idx = self.article_index.get(num) or self.article_index.get(str(_num_to_int(num)))
            if idx is not None:
                found.append(self.articles[idx])
                print(f"   Lookup direct -> الفصل {self.articles[idx].article_num}")
            else:
                print(f"   الفصل {num} absent de la base")
        return found

    def _extract_cited_nums(self, llm_response):
        """Numéros de فصول effectivement mentionnés dans la réponse du LLM."""
        cited = set()
        for raw in self._CITED_ART_RE.findall(llm_response):
            num = _clean_article_num(raw)
            cited.add(num)
            cited.add(str(_num_to_int(num)))
        return cited

    def ask(self, question, top_k=5):
        print(f"\n{'='*60}")
        print(f"Question : {question}")
        print(f"{'='*60}")

        # 1. Lookup direct
        direct_arts = self._lookup_direct(question)

        # 2. Retrieval filtré (CORRIGÉ : retrieve_filtered)
        semantic = self.retriever.retrieve_filtered(question, top_k=top_k)

        # 3. Fusion
        seen  = {a.article_num for a in direct_arts}
        final = list(direct_arts)
        for r in semantic:
            if r.article.article_num not in seen:
                final.append(r.article)
                seen.add(r.article.article_num)

        if not final:
            return "Aucun article pertinent trouvé."

        # Log avec scores RRF pour transparence
        print(f"\nArticles injectés dans le LLM ({len(final)}) :")
        d_nums  = {a.article_num for a in direct_arts}
        rrf_map = {r.article.article_num: r.rrf_score for r in semantic}
        for art in final:
            src   = "direct" if art.article_num in d_nums else "RAG"
            score = f"RRF={rrf_map.get(art.article_num, 0):.4f}" if src == "RAG" else "direct"
            print(f"   الفصل {art.article_num:>8} [{src}] [{score}] — {art.char_count} chars")

        # 4. Contexte LLM
        ctx_parts = [f"الفصل {a.article_num} :\n{a.text}" for a in final]
        context = "\n\n---\n\n".join(ctx_parts)

        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"السؤال : {question}\n\n"
                    f"الفصول القانونية المرجعية "
                    f"(استخدم فقط ما يتعلق مباشرة بالسؤال) :\n\n{context}"
                ),
            },
        ]

        print("\nGénération de la réponse…")
        resp = self.llm.create_chat_completion(
            messages=messages, max_tokens=1200, temperature=0.1, repeat_penalty=1.1,
        )
        llm_answer = resp["choices"][0]["message"]["content"].strip()

        # 5. Articles réellement cités par le LLM
        cited_nums = self._extract_cited_nums(llm_answer)
        cited_arts = [
            a for a in final
            if a.article_num in cited_nums or str(a.article_num_int) in cited_nums
        ]
        if not cited_arts:
            cited_arts = final  # fallback si le LLM ne cite aucun numéro

        # 6. Formatage de la sortie
        out = [f"\n{'='*60}", "REPONSE DE L'AGENT JURIDIQUE", f"{'='*60}\n", llm_answer,
               f"\n\n{'='*60}", "TEXTES OFFICIELS DES ARTICLES UTILISES", f"{'='*60}"]
        for art in cited_arts:
            out.append(f"\n{'─'*50}")
            out.append(f"الفصل {art.article_num}")
            if art.book:       out.append(f"[{art.book}]")
            if art.section:    out.append(f"[{art.section}]")
            if art.subsection: out.append(f"[{art.subsection}]")
            out.append("")
            out.append(art.text)
        out.append(f"\n{'='*60}\n")

        result = "\n".join(out)
        print(result)
        return result


agent = LawConsultantAgent(retriever, llm, articles, article_index)



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Interface interactive
# ══════════════════════════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

CSS = """<style>
  .loc-title  { font-size:1.4em; font-weight:bold; color:#1a237e;
                text-align:center; margin-bottom:8px; direction:rtl; }
  .loc-sub    { font-size:0.9em; color:#555; text-align:center; margin-bottom:16px; }
  .loc-art    { background:#fff8e1; border-left:4px solid #f9a825;
                border-radius:4px; padding:12px; margin-top:10px; direction:rtl; }
  .loc-num    { color:#b71c1c; font-weight:bold; font-size:1.05em; }
  .loc-spin   { color:#1a237e; font-style:italic; }
</style>"""

title_html = widgets.HTML(CSS + """
<div class="loc-title">المستشار القانوني — قانون الالتزامات والعقود المغربي</div>
<div class="loc-sub">اكتب سؤالك القانوني وسيعرض النظام الفصل الكامل والنص الرسمي</div>
""")

question_input = widgets.Textarea(
    placeholder="مثال: ما هي شروط صحة العقد؟\nمثال: متى يجوز فسخ العقد؟",
    layout=widgets.Layout(width="100%", height="90px"),
)
top_k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1, description="عدد الفصول:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="50%"),
)
submit_btn = widgets.Button(description="إرسال السؤال", button_style="primary",
                            layout=widgets.Layout(width="200px", height="40px"))
clear_btn  = widgets.Button(description="مسح", button_style="warning",
                            layout=widgets.Layout(width="100px", height="40px"))
output_area = widgets.Output(layout=widgets.Layout(width="100%", border="1px solid #ddd",
                             border_radius="8px", padding="8px", min_height="100px"))


def on_submit(b):
    question = question_input.value.strip()
    if not question:
        with output_area:
            clear_output()
            display(HTML('<p style="color:red;text-align:right">الرجاء كتابة سؤال</p>'))
        return

    with output_area:
        clear_output()
        display(HTML('<p class="loc-spin">جارٍ التحليل والبحث في الفصول القانونية…</p>'))

    try:
        result = agent.ask(question, top_k=top_k_slider.value)
    except Exception as exc:
        with output_area:
            clear_output()
            display(HTML(f'<p style="color:red">خطأ : {exc}</p>'))
        return

    def _esc(s):
        return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    # Séparer réponse LLM et textes officiels
    sep = "TEXTES OFFICIELS DES ARTICLES UTILISES"
    parts = result.split(sep)
    llm_part  = parts[0].strip() if parts else result
    arts_part = parts[1].strip() if len(parts) > 1 else ""

    def _clean_sep(text):
        lines = [l for l in text.splitlines() if not re.match(r"^[=\-]{3,}$", l.strip())]
        return "\n".join(lines).strip()

    llm_clean  = _clean_sep(llm_part.replace("REPONSE DE L'AGENT JURIDIQUE", "").strip())
    arts_clean = _clean_sep(arts_part)

    html = [CSS, '<div style="direction:rtl;font-family:\'Amiri\',Arial,serif;">',
            '<div style="background:#e8f5e9;border-radius:8px;padding:14px;margin-bottom:12px;">',
            '<div style="color:#1b5e20;font-weight:bold;font-size:1.1em;margin-bottom:8px;">الإجابة القانونية</div>',
            f'<div style="white-space:pre-wrap;font-size:0.95em;">{_esc(llm_clean)}</div>',
            '</div>',
            '<div style="color:#4a148c;font-weight:bold;font-size:1em;margin:12px 0 8px;">النصوص الرسمية للفصول المستخدمة</div>']

    for block in re.split(r"─{3,}", arts_clean):
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        html += ['<div class="loc-art">',
                 f'<div class="loc-num">{_esc(lines[0].strip() if lines else "")}</div>',
                 f'<div style="margin-top:8px;white-space:pre-wrap;font-size:0.92em;">{_esc(chr(10).join(lines[1:]).strip())}</div>',
                 '</div>']

    html.append('</div>')
    with output_area:
        clear_output()
        display(HTML("".join(html)))


def on_clear(b):
    question_input.value = ""
    with output_area:
        clear_output()


submit_btn.on_click(on_submit)
clear_btn.on_click(on_clear)

display(widgets.VBox(
    [title_html, question_input,
     widgets.HBox([submit_btn, clear_btn, top_k_slider],
                  layout=widgets.Layout(gap="12px", align_items="center", margin="8px 0")),
     output_area],
    layout=widgets.Layout(width="100%", padding="12px"),
))